# Verdict RAG pipeline

This notebook documents and evaluates the grounded competitive-programming tutor. It follows the graduation brief: inspect a corpus, chunk it, embed into a persistent vector store, retrieve evidence, generate with Ollama, cite sources, and evaluate at least ten questions.

The production UI is the stripped Verdict mirror. The notebook is the reproducible RAG evidence and evaluation surface.

In [1]:
from pathlib import Path
import json, re, sys
ROOT = Path.cwd()
if (ROOT / 'backend').exists():
    sys.path.insert(0, str(ROOT / 'backend'))
else:
    ROOT = ROOT.parent
    sys.path.insert(0, str(ROOT / 'backend'))
CORPUS = ROOT / 'backend' / 'data' / 'knowledge'
EVAL = ROOT / 'backend' / 'evaluation' / 'questions.json'
corpus_files = sorted([p for p in CORPUS.rglob('*.md') if p.is_file()])
print('Project root:', ROOT)
print('Corpus files found:', len(corpus_files))


Project root: /home/yousefmsm1/Desktop/ITI_LV2_GRAD/Verdict_RAG_ITI_AI_LV2
Corpus files found: 106


## 1. Load and inspect the corpus

The curated corpus contains practical notes for binary search, two pointers, prefix sums, graph traversal, and dynamic programming. Each file includes the invariant, implementation pattern, complexity, and pitfalls. These focused documents reduce retrieval noise compared with indexing the entire source tree.

In [2]:
documents = []
for path in corpus_files:
    text = path.read_text(encoding='utf-8', errors='ignore')
    documents.append({'path': str(path), 'title': path.stem, 'text': text})
print(f'Successfully loaded {len(documents)} documents.')
for doc in documents[:5]:
    print(f" - {doc['title']}: {len(doc['text'])} characters")


Successfully loaded 106 documents.
 - binary_search: 988 characters
 - algebra__all-submasks: 3760 characters
 - algebra__balanced-ternary: 3226 characters
 - algebra__big-integer: 10141 characters
 - algebra__binary-exp: 14003 characters


## 2. Chunking

Chunks are 1,200 characters with a 160-character overlap (the same settings used by `backend/scripts/ingest.py`). The overlap preserves definitions and complexity notes that cross a heading boundary. Hash-based IDs make ingestion repeatable and safe to rerun.

In [3]:
def chunk_text(text, size=1200, overlap=160):
    clean = re.sub(r'\s+', ' ', text).strip()
    step = max(1, size - overlap)
    return [clean[i:i+size] for i in range(0, len(clean), step) if clean[i:i+size].strip()]
chunks = []
for doc in documents:
    for index, chunk in enumerate(chunk_text(doc['text'])):
        chunks.append({'id': f"{doc['title']}::{index}", 'title': doc['title'], 'chunk': index, 'text': chunk})
print('Total chunks:', len(chunks))
print(chunks[0]['text'][:300])

Total chunks: 1064
# Binary Search and Binary Search on the Answer Binary search applies when a sorted search space or monotonic predicate divides candidates into two contiguous regions. For a predicate `feasible(x)`, determine whether it changes from false to true or true to false, maintain a documented invariant, an


## 3. Persist embeddings in Chroma

The application uses `sentence-transformers/all-MiniLM-L6-v2` through Chroma's embedding function. The persistent directory is `data/chroma`; it is ignored by Git because model and index files are generated artifacts.

In [4]:
from app.core.config import Settings
from app.services.retrieval import Retriever
settings = Settings()
retriever = Retriever(settings.chroma_persist_dir, settings.chroma_collection, settings.embedding_model)
if retriever.count() == 0:
    indexed = retriever.add([x['text'] for x in chunks], [x['id'] for x in chunks], [{
        'source_id': x['id'], 'title': x['title'], 'source': x['title'] + '.md', 'chunk': x['chunk']
    } for x in chunks])
    print('Indexed:', indexed, 'stored:', retriever.count())
else:
    print(f'Vector store ready and loaded from disk: {retriever.count()} chunks persisted.')


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Vector store ready and loaded from disk: 2128 chunks persisted.


## 4. Retrieval checks

A useful retrieval result should name a relevant source and expose enough context for a grounded answer. The production API returns these same records as citations.

In [5]:
def show_retrieval(question, k=3):
    results = retriever.search(question, k)
    for rank, result in enumerate(results, 1):
        print(f"{rank}. {result['metadata'].get('title')} (score={result.get('score')})")
        print(result['document'][:220].replace('\n', ' '), '\n')
    return results
show_retrieval('How do I prove a binary search loop is correct?')

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


1. binary_search (score=0.5114721059799194)
# Binary Search and Binary Search on the Answer Binary search applies when a sorted search space or monotonic predicate divides candidates into two contiguous regions. For a predicate `feasible(x)`, determine whether it  

2. binary_search (score=0.5114721059799194)
# Binary Search and Binary Search on the Answer Binary search applies when a sorted search space or monotonic predicate divides candidates into two contiguous regions. For a predicate `feasible(x)`, determine whether it  

3. data_structures__sqrt-tree (score=0.48216164112091064)
ents of the equal size, and the block on one layer have also equal size (in our case, their size is $2^k = 2^4 = 16$. The blocks cover the array entirely, so the first block covers elements $(0 - 15)$ ($(000000_2 - 00111 



[{'id': 'd011221208ae01c602da540815c20597cfc5f721',
  'document': '# Binary Search and Binary Search on the Answer Binary search applies when a sorted search space or monotonic predicate divides candidates into two contiguous regions. For a predicate `feasible(x)`, determine whether it changes from false to true or true to false, maintain a documented invariant, and move one boundary each iteration. Integer implementations should use an overflow-safe midpoint and finish with the smallest or largest feasible candidate required by the problem. Binary search on the answer does not require an explicitly sorted array. It requires an ordered candidate answer and a monotonic feasibility test. Common signals include minimizing a maximum load, maximizing a minimum distance, or finding the least time needed to produce a target amount. Derive bounds from constraints, prove monotonicity, and compute feasibility without exceeding integer limits. Complexity is normally `O(C log R)`, where `C` is the

## 5. Ten-plus question evaluation

The hand-labeled set includes in-domain algorithm questions and out-of-domain questions. `Hit@k` measures whether the expected topic appears in the retrieved source title. Out-of-domain questions are expected to return no labeled topic and are used to inspect abstention behavior.

In [6]:
questions = json.loads(EVAL.read_text(encoding='utf-8'))
rows = []
for item in questions:
    results = retriever.search(item['question'], 3)
    titles = [r['metadata'].get('title', '').lower().replace('_', ' ') for r in results]
    expected = item.get('expected_source')
    hit = bool(expected and any(expected.replace('_', ' ') in t for t in titles))
    rows.append({**item, 'hit_at_3': hit, 'retrieved': titles})
labeled = [r for r in rows if r.get('expected_source')]
hits_count = sum(r['hit_at_3'] for r in labeled)
print(f"Hit@3: {hits_count}/{len(labeled)} = {hits_count/len(labeled):.1%}")
for row in rows:
    status = 'HIT' if row['hit_at_3'] else ('MISS (OOD)' if not row.get('expected_source') else 'MISS')
    print(f"{row['id']:<10} {status:<12} retrieved: {row['retrieved'][:2]}")


Hit@3: 11/11 = 100.0%
bs-1       HIT          retrieved: ['binary search', 'binary search']
bs-2       HIT          retrieved: ['binary search', 'binary search']
prefix-1   HIT          retrieved: ['prefix sums', 'prefix sums']
prefix-2   HIT          retrieved: ['prefix sums', 'prefix sums']
tp-1       HIT          retrieved: ['two pointers', 'two pointers']
tp-2       HIT          retrieved: ['two pointers', 'two pointers']
graph-1    HIT          retrieved: ['graph  breadth-first-search', 'graph  breadth-first-search']
graph-2    HIT          retrieved: ['graph  search-for-connected-components', 'graph  search-for-connected-components']
dp-1       HIT          retrieved: ['dynamic programming  knapsack', 'dynamic programming  knapsack']
dp-2       HIT          retrieved: ['dynamic programming  longest increasing subsequence', 'dynamic programming  longest increasing subsequence']
cpp-1      HIT          retrieved: ['cpp basics io', 'cpp basics io']
ood-1      MISS (OOD)   retrieved:

## 6. Grounded generation and citation checks

Generation is deliberately constrained to retrieved context. If Chroma returns no context, the Ollama prompt tells the model to state that limitation rather than inventing an answer or citation. The API response exposes `grounded`, `retrieved_context`, `citations`, model name, and latency for inspection.

In [7]:
import asyncio
from app.services.generation import Generator
generator = Generator(settings)
context = retriever.search('When should I use prefix sums?', 3)
try:
    answer = await generator.generate('When should I use prefix sums?', context, 'teach')
except (RuntimeError, SyntaxError):
    import nest_asyncio
    nest_asyncio.apply()
    answer = asyncio.run(generator.generate('When should I use prefix sums?', context, 'teach'))
print(answer)
print('Citation markers present:', bool(re.search(r'\[\d+\]', answer)))


Ollama generation failed: All connection attempts failed


Ollama is unavailable, so here is the retrieved guidance without a generated explanation:

[1] prefix_sums: # Prefix Sums and Difference Arrays A prefix sum stores cumulative values so a static range sum `[l, r]` is answered as `prefix[r + 1] - prefix[l]`. Build `prefix[0] = 0` and `prefix[i + 1] = prefix[i] + a[i]`; this half-open convention reduces boundary mistakes. Preprocessing is `O(n)` and each range query is `O(1)`. For many offline range additions, a difference array records `diff[l] += value` and `diff[r + 1] -= 

[2] prefix_sums: # Prefix Sums and Difference Arrays A prefix sum stores cumulative values so a static range sum `[l, r]` is answered as `prefix[r + 1] - prefix[l]`. Build `prefix[0] = 0` and `prefix[i + 1] = prefix[i] + a[i]`; this half-open convention reduces boundary mistakes. Preprocessing is `O(n)` and each range query is `O(1)`. For many offline range additions, a difference array records `diff[l] += value` and `diff[r + 1] -= 

[3] data_structures__sqrt-tree:

## 7. Failure analysis

Common failure modes are visible and testable: no index, no Ollama process, an out-of-domain question, or an ambiguous problem statement. The service remains healthy when optional dependencies are unavailable and returns a clear Ollama-unavailable answer instead of fabricating content.

In [8]:
ood = next((q for q in questions if not q.get('expected_source')), {'question': 'What is the weather today?'})
ood_results = retriever.search(ood['question'], 3)
print('OOD question:', ood['question'])
print('Retrieved titles:', [r['metadata'].get('title') for r in ood_results])
print('Production behavior:', 'answer with limitation when context is empty; never invent citations')

OOD question: Who won the football match yesterday?
Retrieved titles: ['graph__breadth-first-search', 'graph__breadth-first-search', 'graph__2SAT']
Production behavior: answer with limitation when context is empty; never invent citations


### Reproduce from a terminal

```bash
cd backend
python -m scripts.ingest --input data/knowledge
python -m scripts.evaluate_retrieval --persist-dir data/chroma
pytest -q
```

Start Ollama with `ollama serve` and the configured model before testing generated answers. The FastAPI service is launched with `uvicorn app.main:app --reload --port 8000`.